In [1]:
"""
==============================================================
SCRIPT 02.5
INVENTARIO Y ANÁLISIS DEL GROUND TRUTH
==============================================================

Objetivo
--------
Analizar el Ground Truth bufferizado antes de construir
el dataset de entrenamiento.

Entrada
-------
GroundTruth_SAGAMI_Buffer5m.gpkg

Salidas
-------
Resultados_GT/
│
├── Inventario_GT.xlsx
├── Inventario_GT.csv
├── Nivel1.csv
├── Nivel2.csv
├── Nivel3.csv
├── Poligonos_Criticos.csv
├── Resumen_GT.txt
├── Distribucion_N1.png
├── Distribucion_N2.png
└── Distribucion_N3.png

Autor:
Luis Miguel Gómez Meneses

==============================================================
"""

from pathlib import Path

import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# CONFIGURACIÓN
# ==========================================================

INPUT_FILE = Path("/content/GroundTruth_SAGAMI_Buffer5m.gpkg")

OUTPUT_DIR = Path("Resultados_GT")
OUTPUT_DIR.mkdir(exist_ok=True)

PIXEL_SIZE = 10          # metros
PIXEL_AREA = PIXEL_SIZE ** 2

# ==========================================================
# CARGAR GROUND TRUTH
# ==========================================================

print("=" * 70)
print("CARGANDO GROUND TRUTH")
print("=" * 70)

gdf = gpd.read_file(INPUT_FILE)

print(f"Polígonos : {len(gdf)}")
print(f"CRS       : {gdf.crs}")

# ==========================================================
# CALCULAR ÁREAS
# ==========================================================

gdf["area_m2"] = gdf.area
gdf["area_ha"] = gdf["area_m2"] / 10000

gdf["pixeles_estimados"] = (
    gdf["area_m2"] / PIXEL_AREA
)

# ==========================================================
# FUNCIÓN RESUMEN
# ==========================================================

def resumen_clases(campo):

    resumen = (
        gdf.groupby(campo)
        .agg(
            Poligonos=("id", "count"),
            Area_ha=("area_ha", "sum"),
            Area_media_ha=("area_ha", "mean"),
            Area_min_ha=("area_ha", "min"),
            Area_max_ha=("area_ha", "max"),
            Pixeles=("pixeles_estimados", "sum"),
            Pixeles_promedio=("pixeles_estimados", "mean"),
        )
        .reset_index()
    )

    resumen["Porcentaje_area"] = (
        100 *
        resumen["Area_ha"] /
        resumen["Area_ha"].sum()
    )

    resumen = resumen.sort_values(
        "Area_ha",
        ascending=False
    )

    return resumen

# ==========================================================
# GENERAR TABLAS
# ==========================================================

nivel1 = resumen_clases("clase_n1")
nivel2 = resumen_clases("clase_n2")
nivel3 = resumen_clases("clase_n3")

# ==========================================================
# EXPORTAR CSV
# ==========================================================

nivel1.to_csv(
    OUTPUT_DIR / "Nivel1.csv",
    index=False
)

nivel2.to_csv(
    OUTPUT_DIR / "Nivel2.csv",
    index=False
)

nivel3.to_csv(
    OUTPUT_DIR / "Nivel3.csv",
    index=False
)

# ==========================================================
# INVENTARIO GENERAL
# ==========================================================

inventario = gdf[
    [
        "id",
        "predio",
        "clase_n1",
        "clase_n2",
        "clase_n3",
        "area_ha",
        "pixeles_estimados",
    ]
]

inventario.to_csv(
    OUTPUT_DIR / "Inventario_GT.csv",
    index=False
)

with pd.ExcelWriter(
    OUTPUT_DIR / "Inventario_GT.xlsx"
) as writer:

    inventario.to_excel(
        writer,
        sheet_name="Inventario",
        index=False,
    )

    nivel1.to_excel(
        writer,
        sheet_name="Nivel1",
        index=False,
    )

    nivel2.to_excel(
        writer,
        sheet_name="Nivel2",
        index=False,
    )

    nivel3.to_excel(
        writer,
        sheet_name="Nivel3",
        index=False,
    )

# ==========================================================
# POLÍGONOS CRÍTICOS
# ==========================================================

criticos = gdf[
    gdf["pixeles_estimados"] < 50
].copy()

criticos[
    [
        "id",
        "clase_n1",
        "clase_n2",
        "clase_n3",
        "area_ha",
        "pixeles_estimados",
    ]
].to_csv(
    OUTPUT_DIR / "Poligonos_Criticos.csv",
    index=False
)

# ==========================================================
# GRÁFICAS
# ==========================================================

plt.figure(figsize=(9,5))
plt.bar(
    nivel1["clase_n1"],
    nivel1["Pixeles"]
)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Píxeles estimados")
plt.title("Distribución Nivel 1")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Distribucion_N1.png",
    dpi=300
)
plt.close()

plt.figure(figsize=(9,5))
plt.bar(
    nivel2["clase_n2"],
    nivel2["Pixeles"]
)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Píxeles estimados")
plt.title("Distribución Nivel 2")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Distribucion_N2.png",
    dpi=300
)
plt.close()

plt.figure(figsize=(12,5))
plt.bar(
    nivel3["clase_n3"],
    nivel3["Pixeles"]
)
plt.xticks(rotation=90)
plt.ylabel("Píxeles estimados")
plt.title("Distribución Nivel 3")
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "Distribucion_N3.png",
    dpi=300
)
plt.close()

# ==========================================================
# REPORTE
# ==========================================================

area_total = gdf["area_ha"].sum()
pixeles_totales = gdf["pixeles_estimados"].sum()

with open(
    OUTPUT_DIR / "Resumen_GT.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write("=========================================\n")
    f.write("INVENTARIO DEL GROUND TRUTH\n")
    f.write("=========================================\n\n")

    f.write(f"Polígonos             : {len(gdf)}\n")
    f.write(f"Área total (ha)       : {area_total:.2f}\n")
    f.write(f"Píxeles estimados     : {pixeles_totales:.0f}\n")
    f.write(f"Resolución objetivo   : {PIXEL_SIZE} m\n")
    f.write(f"Tamaño de píxel       : {PIXEL_AREA} m²\n\n")

    f.write("=========================================\n")
    f.write("CLASES NIVEL 1\n")
    f.write("=========================================\n")
    f.write(nivel1.to_string(index=False))

    f.write("\n\n=========================================\n")
    f.write("CLASES NIVEL 2\n")
    f.write("=========================================\n")
    f.write(nivel2.to_string(index=False))

    f.write("\n\n=========================================\n")
    f.write("CLASES NIVEL 3\n")
    f.write("=========================================\n")
    f.write(nivel3.to_string(index=False))

    f.write("\n\n=========================================\n")
    f.write("POLÍGONOS CRÍTICOS (<50 píxeles)\n")
    f.write("=========================================\n")

    if len(criticos) == 0:
        f.write("No existen polígonos críticos.\n")
    else:
        f.write(
            criticos[
                [
                    "id",
                    "clase_n3",
                    "pixeles_estimados",
                ]
            ].to_string(index=False)
        )

# ==========================================================
# RESUMEN EN PANTALLA
# ==========================================================

print("\n" + "=" * 70)
print("RESUMEN")
print("=" * 70)

print(f"Área total (ha)          : {area_total:.2f}")
print(f"Píxeles estimados        : {pixeles_totales:.0f}")
print(f"Clases Nivel 1           : {len(nivel1)}")
print(f"Clases Nivel 2           : {len(nivel2)}")
print(f"Clases Nivel 3           : {len(nivel3)}")
print(f"Polígonos críticos (<50) : {len(criticos)}")

print("\nArchivos generados en:")
print(OUTPUT_DIR.resolve())

print("\nProceso finalizado correctamente.")

CARGANDO GROUND TRUTH
Polígonos : 67
CRS       : EPSG:32619


/tmp/ipykernel_975/1838640397.py:252: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()



RESUMEN
Área total (ha)          : 1656.93
Píxeles estimados        : 165693
Clases Nivel 1           : 14
Clases Nivel 2           : 19
Clases Nivel 3           : 36
Polígonos críticos (<50) : 14

Archivos generados en:
/content/Resultados_GT

Proceso finalizado correctamente.
